# AI工学101 — 第36回

## 特徴量エンジニアリング：モデルに何を見せるか

よしレベル、今日はかなり重要だぞ。

これまで僕らは、

```text
データ
↓
前処理
↓
モデル
```

という流れを何度も見てきた。

でも今日は、その中でも根本的な部分。

> **モデルは世界そのものを見ていない。
> 僕らが与えた「表現」を見ている。**

たとえば人間の「年齢」という現実の対象も、

```python
age = 36
```

という数値として与えるのか、

```text
20代
30代
40代
```

というカテゴリとして与えるのか、

あるいは、

```text
成人
高齢者
```

という抽象化された特徴として与えるのかで、モデルが見ている世界は変わる。

つまり特徴量設計は、

> **現実世界 → モデルの内部世界**

への翻訳作業だ。

今日からscikit-learn編の後半戦。

ここで、

```text
「モデルを使う」
```

から、

```text
「モデルが理解できる世界を設計する」
```

へ一段進むぞ。🧠🔧

---

# 🎯 今日のゴール

今日は次を身につける。

* 特徴量とは何かを説明できる
* 数値特徴量とカテゴリ特徴量を区別できる
* One-Hot Encodingを使える
* 欠損値を処理できる
* スケーリングが必要な場面を理解する
* `Pipeline` を使える
* `ColumnTransformer` を使える
* 前処理のリークを防げる
* 特徴量エンジニアリングの基本を理解する

---

# 📖 講義：約20〜25分

# 1. 特徴量とは何か？

機械学習では、

```text
入力データ
```

を、

```text
特徴量
```

として表現する。

例えば家の価格を予測するなら、

```text
面積
築年数
駅からの距離
部屋数
```

など。

```text
現実の家
↓
特徴量ベクトル
```

になる。

例えば、

```python
[
    75,
    10,
    5,
    3
]
```

。

モデルは、

```text
「75平方メートルの家」
```

という概念を直接理解しているわけではない。

単に、

```text
特徴量の値のパターン
```

を見ている。

ここが重要。

---

# 🧠 2. 特徴量は「世界の表現」である

例えば、

```text
気温 30℃
```

という情報がある。

これをそのまま、

```python
temperature = 30
```

として渡せる。

でも、

```python
is_hot = 1
```

に変換することもできる。

あるいは、

```python
temperature_squared = 900
```

という特徴も作れる。

つまり同じ現実でも、

```text
表現方法
```

が複数ある。

```text
現実
↓
特徴量設計
↓
モデルが見る世界
```

。

認知科学っぽく言えば、

> **特徴量はモデルの表象空間の設計図**

でもある。

ここ、レベルの興味とかなり直結する。

---

# 3. 数値特徴量

一番わかりやすい。

例えば、

```text
年齢
身長
売上
気温
距離
```

。

```python
age = 36
height = 151
```

。

数値には、

```text
連続量
```

と、

```text
離散量
```

がある。

---

## 連続量

例えば、

```text
体温
身長
気温
```

。

概念的には、

```text
36.5
36.51
36.511
```

のように連続的な値を取る。

---

## 離散量

例えば、

```text
部屋数
購入回数
クリック数
```

。

```text
0
1
2
3
```

など。

---

# 4. カテゴリ特徴量

例えば、

```text
都道府県
職業
血液型
商品の種類
```

。

```text
東京
京都
大阪
```

など。

ここで問題がある。

モデルは文字列をそのまま理解できないことが多い。

例えば、

```text
Tokyo
Kyoto
Osaka
```

を数値にする必要がある。

---

# 🚨 やってはいけない変換

例えば、

```text
Tokyo = 1
Kyoto = 2
Osaka = 3
```

。

これは危険。

なぜなら、

モデルによっては、

```text
Tokyo < Kyoto < Osaka
```

という意味があるように扱う可能性がある。

つまり、

```text
3
```

が、

```text
1
```

より大きい。

という順序が勝手に生まれる。

しかし、

```text
Tokyo
Kyoto
Osaka
```

には通常、

```text
大小関係
```

はない。

そこで、

```text
One-Hot Encoding
```

を使う。

---

# 5. One-Hot Encoding

例えば、

```text
Tokyo
Kyoto
Osaka
```

。

これを、

```text
Tokyo  Kyoto  Osaka

1      0      0
0      1      0
0      0      1
```

に変換する。

Pythonでは、

```python
from sklearn.preprocessing import OneHotEncoder
```

。

---

# 💻 実習1：カテゴリデータをOne-Hot Encoding

```python
import pandas as pd

df = pd.DataFrame({
    "city": [
        "Tokyo",
        "Kyoto",
        "Osaka",
        "Tokyo"
    ]
})
```

```python
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(
    handle_unknown="ignore"
)

encoded = encoder.fit_transform(
    df[["city"]]
)
```

確認。

```python
print(
    encoded.toarray()
)
```

---

# 🧠 6. handle_unknown="ignore"

これが実務ではかなり重要。

例えば、

学習データ。

```text
Tokyo
Kyoto
Osaka
```

。

でも本番データに、

```text
Nagoya
```

が来た。

すると、

```text
知らないカテゴリ
```

になる。

そこで、

```python
handle_unknown="ignore"
```

。

これにより、

未知のカテゴリが来ても処理が壊れにくくなる。

---

# 7. 欠損値

現実のデータには、

```text
値がない
```

がある。

例えば、

```text
年齢
36
25
NaN
42
```

。

欠損値の扱い方はいくつかある。

---

## 方法1：行を削除

```text
欠損データ
↓
削除
```

。

簡単。

でも、

> データまで失う

可能性がある。

---

## 方法2：平均値で補完

例えば、

```text
10
20
NaN
30
```

。

平均が、

```text
20
```

なら、

```text
10
20
20
30
```

。

---

## 方法3：中央値で補完

```python
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(
    strategy="median"
)
```

中央値は外れ値の影響を受けにくい。

例えば、

```text
10
20
30
10000
```

。

平均は大きく引っ張られる。

中央値は、

```text
25
```

。

---

# 💻 実習2：欠損値補完

```python
df = pd.DataFrame({
    "age": [
        20,
        30,
        None,
        50
    ]
})
```

```python
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(
    strategy="median"
)

age_imputed = imputer.fit_transform(
    df[["age"]]
)
```

確認。

```python
print(age_imputed)
```

---

# 🚨 ここでリーク注意

絶対に、

```text
train + test
↓
まとめて平均計算
```

しない。

なぜなら、

```text
テストデータの情報
```

を使ってしまうから。

正しい流れは、

```text
train
↓
中央値を計算
↓
trainを変換

test
↓
trainで計算した中央値を使う
```

。

つまり、

```python
imputer.fit(X_train)
```

して、

```python
imputer.transform(X_train)
imputer.transform(X_test)
```

。

---

# 8. スケーリング

例えば、

```text
年齢
0〜100

年収
0〜10,000,000
```

。

そのままだと、

```text
年収
```

の数値スケールが圧倒的に大きい。

そこで、

```text
スケーリング
```

をする。

---

# StandardScaler

```python
from sklearn.preprocessing import StandardScaler
```

。

概念的には、

```text
平均
↓
0

標準偏差
↓
1
```

。

---

# 💻 実習3：StandardScaler

```python
from sklearn.preprocessing import StandardScaler

X = pd.DataFrame({
    "age": [20, 30, 40],
    "income": [3_000_000, 5_000_000, 10_000_000]
})
```

```python
scaler = StandardScaler()

X_scaled = scaler.fit_transform(
    X
)
```

確認。

```python
print(X_scaled)
```

---

# 🧠 9. スケーリングが重要なモデル

特に、

```text
KNN
KMeans
SVM
Logistic Regression
Neural Network
PCA
```

など。

距離や勾配、数値の大きさが重要になるモデルでは注意。

---

# 逆に木系モデル

例えば、

```text
RandomForest
DecisionTree
Gradient Boosting
```

。

これらは一般に、

```text
スケーリングの必要性は低い
```

。

なぜなら、

```text
特徴量 < 10
```

のような閾値で分割するため。

例えば、

```text
10万円
```

を、

```text
0.5
```

にスケールしても、

順序自体は変わらない。

---

# 10. Pipeline

ここから超重要。

前処理とモデルを別々に書くと、

```text
コードが長い
処理漏れ
リーク
CVとの整合性
```

などが起きる。

そこで、

```python
Pipeline
```

。

---

# 💻 実習4：Pipeline

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
```

```python
pipe = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        LogisticRegression()
    )
])
```

学習。

```python
pipe.fit(
    X_train,
    y_train
)
```

予測。

```python
pred = pipe.predict(
    X_test
)
```

。

このPipelineは、

```text
入力
↓
StandardScaler
↓
LogisticRegression
↓
予測
```

をまとめて管理する。

---

# 🧠 11. Pipelineの利点

重要なのは、

```text
fit
```

と、

```text
transform
```

の流れを管理できること。

CVの中でも、

```text
foldごとに
前処理をfit
```

できる。

つまり、

> **Pipelineはコードを短くするだけではない。
> 実験設計の安全装置でもある。**

---

# 12. 数値列とカテゴリ列を別々に処理する

現実のデータは、

```text
年齢
収入
都市
職業
```

のように混ざっている。

例えば、

```text
age
income
↓
数値処理

city
job
↓
カテゴリ処理
```

。

ここで、

```python
ColumnTransformer
```

を使う。

---

# 💻 実習5：ColumnTransformer

```python
from sklearn.compose import ColumnTransformer
```

```python
numeric_features = [
    "age",
    "income"
]

categorical_features = [
    "city",
    "job"
]
```

前処理。

```python
preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                )
            ),
            (
                "scaler",
                StandardScaler()
            )
        ]),
        numeric_features
    ),

    (
        "cat",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                )
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore"
                )
            )
        ]),
        categorical_features
    )
])
```

これで、

```text
数値列
↓
欠損補完
↓
StandardScaler

カテゴリ列
↓
欠損補完
↓
One-Hot
```

を同時にできる。

---

# 13. 最終的な完全Pipeline

```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
```

```python
model = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])
```

これで、

```text
生データ
↓
数値処理
カテゴリ処理
↓
特徴量生成
↓
モデル
↓
予測
```

が一本化される。

---

# 💻 実習6：本格的なデータセットで試す

今日は、

```python
from sklearn.datasets import fetch_openml
```

ではなく、まずローカルで扱いやすいデータを使ってもいい。

たとえば簡単な人工データを作る。

```python
df = pd.DataFrame({
    "age": [
        25,
        40,
        35,
        None,
        50,
        30
    ],

    "income": [
        300,
        500,
        450,
        600,
        None,
        400
    ],

    "city": [
        "Kyoto",
        "Tokyo",
        "Kyoto",
        "Osaka",
        "Tokyo",
        None
    ],

    "bought": [
        0,
        1,
        1,
        1,
        0,
        0
    ]
})
```

特徴量。

```python
X = df.drop(
    columns="bought"
)
```

正解ラベル。

```python
y = df["bought"]
```

分割。

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42
)
```

そして、

```python
model.fit(
    X_train,
    y_train
)
```

予測。

```python
pred = model.predict(
    X_test
)
```

。

---

# 🧠 14. 特徴量エンジニアリング

特徴量エンジニアリングは、

> **モデルにとって有用な表現を作ること**

。

例えば、

```text
生年月日
```

より、

```text
年齢
```

の方が直接使いやすい場合がある。

---

例えば、

```text
2026-08-24
```

という日付。

そのままより、

```text
年
月
曜日
祝日か
月末か
```

に分ける。

```text
2026
08
Monday
False
False
```

。

同じデータでも、

```text
モデルが利用できる構造
```

を作れる。

---

# 🧠 15. Interaction

例えば、

```text
広告表示回数
```

だけ。

あるいは、

```text
年齢
```

だけ。

単独では弱いかもしれない。

でも、

```text
年齢 × 広告表示回数
```

という関係に意味があるかもしれない。

これが、

```text
interaction
```

。

---

例えば、

```text
広告を10回見た

20歳
↓
購入

80歳
↓
購入しない
```

。

単純なモデルでは、

```text
年齢
```

と、

```text
広告回数
```

を別々に見るだけかもしれない。

でも、

```text
年齢 × 広告回数
```

を作ることで、

> 「年齢によって広告回数の効果が違う」

を表現できる。

---

# 16. Feature Engineeringは魔法ではない

ここも重要。

特徴量を100個作れば強くなる、

わけではない。

むしろ、

```text
ノイズ
↓
過学習
```

が起きる可能性もある。

だから、

```text
特徴量を作る
↓
評価
↓
CV
↓
重要度確認
```

が必要。

---

# 👾 今日のボス戦

## 問題

あるECサイト。

データには、

```text
customer_id
age
city
income
purchase_count
last_login
```

がある。

購入予測モデルを作る。

さて。

---

## customer_id

例えば、

```text
100023
100024
100025
```

。

これは普通、

```text
数値だから連続値
```

として扱ってはいけない。

IDの大小に意味がないから。

---

## age

数値。

```text
欠損
↓
中央値補完
```

などを検討。

---

## city

カテゴリ。

```text
One-Hot
```

。

---

## income

数値。

欠損補完。

必要に応じて、

```text
StandardScaler
```

。

---

## purchase_count

数値。

ただし、

```text
極端に歪んだ分布
```

なら、

```text
log変換
```

も検討。

---

## last_login

日付。

そのままではなく、

```text
現在から何日前か
曜日
月
```

などを作れる。

---

# 💻 演習

## 演習1

次のデータを分類してください。

```text
user_id
age
prefecture
salary
subscription_plan
```

```text
数値
カテゴリ
ID
```

に分類する。

---

## 演習2

`prefecture` を、

```text
Tokyo = 1
Kyoto = 2
Osaka = 3
```

にすることの問題を説明する。

---

## 演習3

次のデータで、

```text
train
↓
中央値 = 30
```

。

testに、

```text
NaN
```

がある。

何で補完する？

```text
A. train + testの中央値

B. trainの中央値
```

答えはもちろん、

> **B**

。

---

## 演習4

次のPipelineを自分で組む。

```text
数値列
↓
SimpleImputer
↓
StandardScaler

カテゴリ列
↓
SimpleImputer
↓
OneHotEncoder

全体
↓
LogisticRegression
```

。

---

# 🧠 今日の核心

今日のAI工学101の核心は、

> **特徴量は単なるデータ列ではない。
> モデルにとっての「世界の表現」である。**

例えば、

```text
人間
↓
感覚器
↓
脳内表象
↓
判断
```

。

機械学習なら、

```text
現実世界
↓
データ収集
↓
特徴量表現
↓
モデル
↓
予測
```

。

この二つはかなり似ている。

もちろん人間の認知を単純な特徴量処理に還元するわけではない。

でも、

> **何を入力として表現するかが、その後に可能な推論を制約する**

という意味では重要な共通点がある。

今日の内容は、単なるscikit-learnのAPI暗記ではない。

> **表現が変われば、モデルが見る構造も変わる。**

これはAI工学でも、認知科学でも、めちゃくちゃ重要な感覚だぞ。

---

# 🧭 AI工学101・現在地

```text
Python基礎
↓
NumPy
↓
scikit-learn

教師あり学習
↓
分類
回帰

モデル評価
↓
CV
Metrics
Leakage

時系列
↓
Concept Drift

教師なし学習
↓
Clustering
PCA
t-SNE
UMAP
Anomaly Detection

そして今日
↓
Feature Engineering
↓
モデルに何を見せるか
```

scikit-learn編、かなり工学的な領域に入ってきた。

モデル選びより前に、

```text
データはどうなっている？
何を表現している？
モデルに何を見せる？
前処理は安全？
```

を考える。

ここまで来ると、だいぶ**「とりあえずモデルを呼ぶ人」から「機械学習システムを設計する人」**へ寄ってきてるぞ。ふふふ💪🧠

---

# 🔜 第37回

## 回帰モデルを本気で比較する：Linear → Random Forest → Gradient Boosting

次回は、

```text
同じデータ
↓
複数モデル
↓
公平に比較
```

をやる。

扱うのは、

* Linear Regression
* Ridge / Lasso
* Random Forest Regressor
* Gradient Boosting
* MAE / RMSE / R²
* Cross Validation
* Baseline
* ハイパーパラメータ
* 「複雑なモデル＝強い」ではない理由

テーマは、

> **「良いモデル」は絶対的な性能ではなく、問題・データ・評価方法との関係で決まる。**

いよいよ次は、**モデル比較を実験として設計する力**を鍛えるぞ。ナデナデしながら進軍だ、レベル💖🔧🧠